# HypatiaX — Unified Experiment Notebook  v7
**Paper:** *HypatiaX: A Hybrid Symbolic-Neural Framework for Extrapolation-Reliable Analytical Discovery*  
**Author:** Ruperto Pedro Bonet Chaple · **Venue:** JMLR v3.0, April 2026  
**LLM component:** Claude (`claude-sonnet-4-20250514`) via Anthropic API — LLM results are stochastic  
**NN / PySR:** exactly reproducible at seed 42

> **v7 changes (synced with run_all_checkpoint.py v4.8):** `claude-sonnet-4-20250514` canonical model string; `ONE_EQUATION` / `--one-equation` smoke-test wired to `FAST=1` branch; `fixup-init` step added to 0-C setup; `N_TASKS_INSTABILITY` corrected to 70 (FIX-T1); `DEFI_V3C_NO_TIMEOUT_FLAGS` / `DEFI_TASK_FILTER` / `DEFI_SEEDS` / `SKIP_PKG_CHECK` / `HYPATIAX_CORE_OPTIONAL` env vars added where matching run_all steps set them; `VERIFY_RESULTS_DIR` / `PATCHED_DATA_DIR` propagated to verify cell (FIX-VERIFY); `TABLE_OUTDIR` propagated to tables cell (FIX-TABLES); NB-06 pre-audit block aligned with run_all.

| # | Experiment | Protocol | Paper section | Expected result |
|---|---|---|---|---|
| Exp 1  | DeFi 74-task benchmark v3.0      | `experiment_protocol_ablation_exp1.py`             | §10.2–10.4, §10.6 | 89.2 % R²>0.99 · 0 catastrophic · 1.73× speedup |
| Exp 1b | Portfolio Variance seed sweep     | `experiment_protocol_defi_v3.py`                   | §10.5             | P(H>P) ≈ 0.76 across seeds 42/99/123/777/2024   |
| Exp 2  | Feynman 30-equation extrapolation | `experiment_protocol_feynman_exp2.py`              | §10.7             | 9/30 (30 %)  *(4–8 h)*                          |
| Exp 3  | Nguyen-12 SR suite (seed 42)      | `experiment_protocol_nguyen12_exp3.py`             | §10.8             | 11/12 H (91.7 %)                                |
| Exp 3b | Nguyen-12 stability (seed 123)    | `experiment_protocol_nguyen12_exp3.py --seed 123`  | §10.8 stability   | consistent with seed 42                         |
| Supp B | Noise/sample-complexity sweep     | `experiment_protocol_noise_sweep.py`               | Supp B            | EHD 100 % at all σ · plateau ≈ N=500            |
| Supp A | Hybrid routing improvements       | `experiment_protocol_hybrid_routing.py`            | Supp A §1–8       | +6 pp Fix1, +5 pp Fix2, +1 pp Fix3              |
| §10.9  | Instability analysis (K=30)       | `experiment_protocol_instability_rf02_04.py`       | §10.9             | Spearman ρ=−0.70, p<0.001                       |
| §10.8c | Extrapolation comparative         | `experiment_protocol_extrapolation_comparative.py` | §10.8             | cross-method R² across OOD regimes              |
| §11a   | Provenance audit (orchestration)  | `experiment_protocol_provenance_audit.py`          | §11               | run after all experiments                       |
| §11b   | Result-family linker              | `discover_provenance.py`                           | §11               | links result files → families → paper tables    |
| §11c   | Internal import DAG               | `scan_internal_imports.py`                         | §11               | logs/repro_output/import_graph.dot              |

| §12a   | Bib audit                         | `NB-01_Citation_Bibliography_Audit.ipynb`          | §B    | FIX-B1/B2/B3 — missing/dup bibitems            |
| §12b   | Cross-reference audit              | `NB-02_CrossReference_Label_Audit.ipynb`           | §B    | FIX-XR1–XR4 — undef refs, dup labels            |
| §12c   | Section structure audit            | `NB-03_Section_Structure_Numbering.ipynb`          | §B    | diagnostic — section tree & equation count      |
| §12d   | Numerical consistency audit        | `NB-04_Numerical_Consistency_Checker.ipynb`        | §B    | FIX-N1/N2/N3 — 70 vs 71, terminology           |
| §12e   | Figure dependency audit            | `NB-05_Figure_Image_Dependency_Checker.ipynb`      | §B    | FIX-F1–F4 — missing figures, fbox placeholders |
| §12f   | Code quality pre-audit             | `NB-06_Code_Quality_Pipeline_Integrity.ipynb`      | §B    | FIX-C1/C3 — dup case names, split mismatch     |

> **Run order:** cells are ordered to match the pipeline. Run top-to-bottom once.  
> For Colab: skip SSH cells (1-A / 1-B / Step 2) if using the public repo.


> **PAPER-QUALITY VERSION** — `FAST=0` hardcoded. Full run ~15-25 h. Colab Pro+ or Kaggle + skip-slow recommended.


## Platform Setup — Colab / Kaggle
> Run this cell **first**, before any other cell. Sets `FAST=0` (paper-quality) and mounts Drive on Colab.


In [ ]:
# ── Platform detection & FAST=0 override ─────────────────────────────────
import os
import sys

# PAPER-QUALITY: force FAST=0 unconditionally
os.environ['FAST'] = '0'

# Seeds (do not change for paper reproducibility)
os.environ.setdefault('NN_SEED',    '42')
os.environ.setdefault('PYSR_SEED',  '42')
os.environ.setdefault('PYTHONHASHSEED', '42')

# Detect platform
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Platform : Google Colab')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Drive    : mounted at /content/drive')
    except Exception as e:
        print(f'Drive    : skipped ({e})')
    os.system('pip install -q --upgrade pip')
elif IS_KAGGLE:
    print('Platform : Kaggle')
    print('NOTE     : 12-hour hard limit. Full run needs --skip-slow + checkpoint.')
else:
    print('Platform : Local / other')

# Optional: uncomment to skip the three slow experiments (fits Kaggle 12 h)
# os.environ['SKIP_SLOW'] = '1'  # skips Feynman(exp2), SuppB, Instability

print(f'FAST     : {os.environ["FAST"]} (0 = paper-quality)')
print(f'Seed     : NN={os.environ["NN_SEED"]} / PYSR={os.environ["PYSR_SEED"]}')


## GPU / Resource Check
> Verifies available accelerator. PySR benefits from more CPU cores; GPU helps NN training.


In [ ]:
import multiprocessing
import os
import subprocess


def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip()

print(f'CPU cores : {multiprocessing.cpu_count()}')

try:
    mem = _run("grep MemTotal /proc/meminfo | awk '{print $2}'")
    gb = round(int(mem) / 1024**2, 1) if mem else '?'
    print(f'RAM (GB)  : {gb}')
except Exception:
    print('RAM       : unknown')

try:
    gpu = _run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null')
    print(f'GPU       : {gpu}' if gpu else 'GPU       : none (CPU-only — PySR slower)')
except Exception:
    print('GPU       : check skipped')

print('Tip: Colab → Runtime > Change runtime type > T4 GPU')
print('     Kaggle → right panel > Session options > Accelerator > GPU T4 x2')


## 0-A · Clone repo & install dependencies
> Run the code cell immediately below to clone **`LLM-HypatiaX-PAPERS-Public`**
> via plain HTTPS (no GitHub account needed) and install all Python dependencies.
> If the repo is already present locally, the cell is a no-op.
>
> The three Raw cells that follow (1-A / 1-B / Step 2) contain the original SSH
> setup for the private repo. **Public-repo users: leave them as Raw and skip to 0-B.**


## 🌐 Public repo — no SSH or GitHub account required

> This notebook targets **`sednabcn/LLM-HypatiaX-PAPERS-Public`** — a fully
> public repository. No SSH key, no GitHub account, and no Personal Access
> Token are needed.

**What to do:**

1. **Skip cells 1-A, 1-B, and Step 2** (SSH key setup — not needed for the public repo).
2. Run the **0-A clone cell** below to download the repository via plain HTTPS.
3. Continue from **0-B** onward.

---
> **Private-repo users (JMLR reviewers with collaborator access):** the three
> Raw cells below (1-A · 1-B · Step 2) contain the original SSH setup. Convert
> them to Code, follow the four-step instructions printed inside, then replace
> `PUBLIC_REPO` in the clone cell with `LLM-HypatiaX-PAPERS` and re-run.


### 1-A · Generate key &nbsp;⬆️ &nbsp;&nbsp;|&nbsp;&nbsp; 1-B · Restore key &nbsp;⬇️
> **Every session:** run only **1-B**. Run **1-A** once if you have no key yet.


In [5]:
# ── 0-A · Clone public repo (HTTPS, no authentication required) ──────────
import os
import subprocess
import sys
from pathlib import Path

GITHUB_USER = "sednabcn"
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"   # public — no token needed
src = Path(f"../{GITHUB_REPO}")

# Install runtime dependencies
subprocess.run(
    ["pip", "install", "-q",
     "anthropic", "pysr", "pint", "python-dotenv", "pyyaml",
     "scikit-learn", "numpy", "pandas"],
    check=True
)
print("✓ Dependencies installed")

if src.exists():
    print("✓ Source repo already present at", src.resolve())
else:
    https_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print(f"Cloning {https_url} ...")
    r = subprocess.run(
        ["git", "clone", https_url, str(src)],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print("✓ Cloned to", src.resolve())
    else:
        raise RuntimeError(
            f"Clone failed:\n{r.stderr.strip()}\n\n"
            "If you are behind a proxy, set HTTPS_PROXY and retry.\n"
            "Manual alternative:\n"
            f"  git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {src}"
        )


Trying SSH clone...
✓ Cloned via SSH to /LLM-HypatiaX-PAPERS


## 0-B · Verify source tree
> Confirms every file the benchmarks import is present before spending time running.


In [6]:
from pathlib import Path

# ── Repo root (public name) ───────────────────────────────────────────────
REPO_ROOT = Path("../LLM-HypatiaX-PAPERS-Public")

# hypatiax package: at repo root (public) or nested papers/2025-JMLR/ (private)
HYPATIAX_SRC_PUBLIC  = REPO_ROOT / "hypatiax"
HYPATIAX_SRC_PRIVATE = REPO_ROOT / "papers" / "2025-JMLR" / "hypatiax"

if HYPATIAX_SRC_PUBLIC.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PUBLIC
    print("Layout : PUBLIC  (hypatiax/ at repo root)")
elif HYPATIAX_SRC_PRIVATE.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PRIVATE
    print("Layout : PRIVATE (hypatiax/ nested under papers/2025-JMLR/)")
else:
    raise SystemExit(
        f"HypatiaX source not found.\n"
        f"  checked: {HYPATIAX_SRC_PUBLIC.resolve()}\n"
        f"  checked: {HYPATIAX_SRC_PRIVATE.resolve()}\n"
        "Run the clone cell above first."
    )

print(f"HYPATIAX_SRC : {HYPATIAX_SRC.resolve()}\n")

# ── Files checked relative to HYPATIAX_SRC (hypatiax-internal) ───────────
hypatiax_files = [
    # Engine
    "tools/symbolic/hybrid_system_v50_2.py",
    "tools/symbolic/symbolic_engine.py",
    "tools/symbolic/physics_aware_regressor.py",
    "tools/symbolic/smart_structure_detector.py",
    # Validation
    "tools/validation/dimensional_validator.py",
    "tools/validation/domain_validator.py",
    "tools/validation/ensemble_validator.py",
    "tools/validation/symbolic_validator.py",
    # hypatiax/protocols/ library modules
    "protocols/experiment_protocol_defi.py",
    "protocols/experiment_protocol_defi_20.py",
    "protocols/experiment_protocol_nguyen12.py",
    "protocols/experiment_protocol_all_18_a.py",
    "protocols/experiment_protocol_all_20.py",
    "protocols/experiment_protocol_all_30.py",
    "protocols/experiment_protocol_benchmark.py",
    "protocols/experiment_protocol_benchmark_v2.py",
    "protocols/experiment_protocol_comparative.py",
    # Analysis & baselines
    "experiments/comparison/test_suite_comparative_v3.py",
    "experiments/tests/extrapolation_test_protocol.py",
    "analysis/statistical_analysis.py",
    "core/training/baseline_neural_network.py",
    "core/base_pure_llm/baseline_pure_llm.py",
]

# ── Files checked relative to REPO_ROOT (repo-level) ─────────────────────
repo_files = [
    # Root-level protocol runners
    "protocols/experiment_protocol_ablation_exp1.py",
    "protocols/experiment_protocol_defi_v3.py",
    "protocols/experiment_protocol_feynman_exp2.py",
    "protocols/experiment_protocol_nguyen12_exp3.py",
    "protocols/experiment_protocol_noise_sweep.py",
    "protocols/experiment_protocol_hybrid_routing.py",
    "protocols/experiment_protocol_instability_rf02_04.py",
    "protocols/experiment_protocol_extrapolation_comparative.py",
    "protocols/experiment_protocol_provenance_audit.py",
    # Provenance & import-graph tools
    "discover_provenance.py",
    "scan_internal_imports.py",
    # Paper audit notebooks
    "notebooks/NB-01_Citation_Bibliography_Audit.ipynb",
    "notebooks/NB-02_CrossReference_Label_Audit.ipynb",
    "notebooks/NB-03_Section_Structure_Numbering.ipynb",
    "notebooks/NB-04_Numerical_Consistency_Checker.ipynb",
    "notebooks/NB-05_Figure_Image_Dependency_Checker.ipynb",
    "notebooks/NB-06_Code_Quality_Pipeline_Integrity.ipynb",
]
# provenance_map.json is optional (warn, not error)
optional_repo_files = ["provenance_map.json"]

found = 0
total = len(hypatiax_files) + len(repo_files)

print("── hypatiax-internal files ──────────────────────────────────")
for f in hypatiax_files:
    ok = (HYPATIAX_SRC / f).exists()
    print(f"  {'✓' if ok else '✗'}  {f}")
    if ok: found += 1

print("\n── repo-root files ──────────────────────────────────────────")
for f in repo_files:
    ok = (REPO_ROOT / f).exists()
    print(f"  {'✓' if ok else '✗'}  {f}")
    if ok: found += 1

print("\n── optional files (warn only) ────────────────────────────────")
for f in optional_repo_files:
    ok = (REPO_ROOT / f).exists()
    print(f"  {'✓' if ok else '⚠ (optional)'}  {f}")

missing = total - found
print(f"\n{found}/{total} critical files present", end="")
if found == total:
    print("  ✓")
else:
    print(f"  ⚠  {missing} missing — check source repo.")


Source root: /LLM-HypatiaX-PAPERS/papers/2025-JMLR/hypatiax

  ✓  tools/symbolic/hybrid_system_v50_2.py
  ✓  tools/symbolic/symbolic_engine.py
  ✓  tools/symbolic/physics_aware_regressor.py
  ✓  tools/symbolic/smart_structure_detector.py
  ✓  tools/validation/dimensional_validator.py
  ✓  tools/validation/domain_validator.py
  ✓  tools/validation/ensemble_validator.py
  ✓  tools/validation/symbolic_validator.py
  ✓  experiments/benchmarks/hypatiax_defi_benchmark_v3c.py
  ✓  experiments/benchmarks/run_comparative_suite_benchmark_v2.py
  ✓  experiments/benchmarks/run_dual_condition_benchmark.py
  ✓  experiments/benchmarks/run_dual_sweep_benchmarks.py
  ✓  experiments/benchmarks/run_hybrid_system_benchmark.py
  ✓  experiments/benchmarks/run_noise_sweep_benchmark.py
  ✓  experiments/benchmarks/run_sample_complexity_benchmark.py
  ✓  experiments/comparison/test_suite_comparative_v3.py
  ✓  experiments/tests/extrapolation_test_protocol.py
  ✗  analysis/statistical_analysis_unified.py
  ✓  co

## 0-C · Set `sys.path` and environment variables
> Run once. All subsequent cells inherit these settings.


In [ ]:
import os
import sys
from pathlib import Path

# ── Repo root ─────────────────────────────────────────────────────────────
REPO_BASE  = Path("../LLM-HypatiaX-PAPERS-Public")
REPRO_ROOT = str(REPO_BASE.resolve())

# ── hypatiax package location (public: repo root / private: papers/2025-JMLR/) ─
HYPATIAX_SRC_PUBLIC  = REPO_BASE / "hypatiax"
HYPATIAX_SRC_PRIVATE = REPO_BASE / "papers" / "2025-JMLR" / "hypatiax"

if HYPATIAX_SRC_PUBLIC.exists():
    HYPATIAX_SRC       = str(HYPATIAX_SRC_PUBLIC.resolve())
    HYPATIAX_IMPORT_ROOT = str(REPO_BASE.resolve())
elif HYPATIAX_SRC_PRIVATE.exists():
    HYPATIAX_SRC         = str(HYPATIAX_SRC_PRIVATE.resolve())
    HYPATIAX_IMPORT_ROOT = str((REPO_BASE / "papers" / "2025-JMLR").resolve())
else:
    raise SystemExit(
        f"HypatiaX source not found.\n"
        f"  checked: {HYPATIAX_SRC_PUBLIC.resolve()}\n"
        f"  checked: {HYPATIAX_SRC_PRIVATE.resolve()}\n"
        "Run the clone cell (0-A) first."
    )

# ── sys.path ──────────────────────────────────────────────────────────────
for p in [HYPATIAX_IMPORT_ROOT, REPRO_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── environment variables ─────────────────────────────────────────────────
os.environ["REPRO_ROOT"]           = REPRO_ROOT
os.environ["HYPATIAX_ROOT"]        = HYPATIAX_SRC
os.environ["HYPATIAX_IMPORT_ROOT"] = HYPATIAX_IMPORT_ROOT
os.environ["PYTHONPATH"] = os.pathsep.join(
    [HYPATIAX_IMPORT_ROOT, REPRO_ROOT]
    + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
)
os.environ["PYTHON_JULIACALL_HANDLE_SIGNALS"] = "yes"

# ── Change CWD to repo root so all subsequent !python calls resolve ───────
# (Jupyter !python runs from the kernel's CWD, not the notebook location)
os.chdir(REPRO_ROOT)
print("CWD changed to:", os.getcwd())

print("REPRO_ROOT          :", REPRO_ROOT)
print("HYPATIAX_SRC        :", HYPATIAX_SRC)
print("HYPATIAX_IMPORT_ROOT:", HYPATIAX_IMPORT_ROOT)

harness = [
    "protocols/_base.py", "protocols/universal_protocol.py",
    "core/runners/common.py",
    "scripts/patches/apply_patches.py", "reproducibility/hash_lock.py",
    "config/repro.yaml",
]
# Verify config/repro.yaml has the correct model string
import yaml as _yaml

repro_cfg = Path(REPRO_ROOT, "config", "repro.yaml")
if repro_cfg.exists():
    try:
        cfg = _yaml.safe_load(repro_cfg.read_text())
        model = cfg.get("llm_model", "")
        _VALID_MODELS = {"claude-sonnet-4-20250514"}
        if model not in _VALID_MODELS:
            print(f"  ⚠  config/repro.yaml llm_model={model!r} — expected one of {_VALID_MODELS}")
            print("     Fix: set llm_model: claude-sonnet-4-20250514  in config/repro.yaml")
        else:
            print(f"  ✓  config/repro.yaml llm_model={model!r}")
    except Exception:
        print("  ⚠  could not parse config/repro.yaml")
print("\nRepro harness:")
for f in harness:
    print(f"  {'✓' if Path(REPRO_ROOT, f).exists() else '✗'}  {f}")

# ── v7: fixup-init (FIX-INIT-PY) — guard broken HypatiaX import ──────────
# Mirrors run_all_checkpoint.py fixup-init step.  Wraps the bare
# `from hypatiax.core import HypatiaX` line in hypatiax/__init__.py inside
# a try/except so sub-packages (hypatiax.protocols.*, etc.) remain
# importable even when hypatiax.core is broken or not compiled.
import pathlib as _pl_fi
import subprocess as _sub_fi
import sys as _sys_fi

_fi_init = _pl_fi.Path(REPRO_ROOT) / 'hypatiax' / '__init__.py'
if _fi_init.exists():
    _fi_src = _fi_init.read_text(encoding='utf-8')
    _BAD  = 'from hypatiax.core import HypatiaX'
    _GOOD = ('try:\n'
             '    from hypatiax.core import HypatiaX  # noqa: F401\n'
             'except Exception:  # broken core does not block sub-packages\n'
             '    HypatiaX = None  # type: ignore')
    if _BAD in _fi_src and 'except Exception:' not in _fi_src:
        _fi_init.write_text(_fi_src.replace(_BAD, _GOOD), encoding='utf-8')
        print('  ✓ fixup-init: hypatiax/__init__.py patched — HypatiaX import guarded')
    else:
        print('  ✓ fixup-init: already patched or BAD string absent')
else:
    print('  ⚠ fixup-init: hypatiax/__init__.py not found — skipping')


REPRO_ROOT          : /content
HYPATIAX_SRC        : /LLM-HypatiaX-PAPERS/papers/2025-JMLR/hypatiax
HYPATIAX_IMPORT_ROOT: /LLM-HypatiaX-PAPERS/papers/2025-JMLR

Repro harness:
  ✗  protocols/_base.py
  ✗  protocols/universal_protocol.py
  ✗  core/runners/common.py
  ✗  core/runners/run_dual_condition_benchmark.py
  ✗  scripts/patches/apply_patches.py
  ✗  reproducibility/hash_lock.py


## 1 · API key (Anthropic Claude)


In [ ]:
import os
from pathlib import Path

# Option 1 — set directly (do not commit with a real key)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Option 2 — .env file at repo root (recommended for local runs)
try:
    from dotenv import load_dotenv
    load_dotenv(Path(".env"))
    print(".env loaded")
except ImportError:
    pass

# Option 3 — Google Colab secrets
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Colab userdata loaded")
except Exception:
    pass

# Option 4 — Kaggle UserSecrets (Kaggle notebook environment)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    print("Kaggle UserSecrets loaded")
except Exception:
    pass

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set.\n"
        "Colab: Secrets panel → add ANTHROPIC_API_KEY.\n"
        "Kaggle: Add-ons → Secrets → add ANTHROPIC_API_KEY.\n"
        "Local: export ANTHROPIC_API_KEY='sk-ant-...' or add to .env"
    )
print(f"API key set: YES ({len(key)} chars)")


## 2 · Runtime configuration
Mirrors `config/repro.yaml`. Set once; all `!python` calls below inherit via environment.


In [ ]:
# ── Runtime config — PAPER-QUALITY (FAST=0 hardcoded) ──────────────────
# This notebook always runs in paper-quality mode.
# For smoke-tests use HypatiaX_Experiments_v7_PUBLIC_fast.ipynb
import os

FAST = 0  # hardcoded — do NOT change
os.environ['FAST'] = '0'

# Seeds & model
os.environ.setdefault('NN_SEED',    '42')
os.environ.setdefault('PYSR_SEED',  '42')
os.environ.setdefault('PYTHONHASHSEED', os.environ.get('PYSR_SEED', '42'))
os.environ.setdefault('LLM_MODEL', 'claude-sonnet-4-20250514')
os.environ['LLM_RETRIES']  = '3'
os.environ['LLM_K_RUNS']   = '1'
os.environ['ENGINE_NAME']  = 'hybrid_system_v50_2'

# Task counts — FULL paper values
os.environ['N_TASKS_DEFI']        = '74'
os.environ['N_TASKS_INSTABILITY'] = '70'

# Neural network
os.environ['PCA_TRAIN_FRAC']  = '0.40'
os.environ['NN_TIME_LIMIT']   = '120'

# PySR — paper-quality values
os.environ['PYSR_NITERATIONS'] = '1000'
os.environ['PYSR_POPULATIONS'] = '30'
os.environ['PYSR_TIMEOUT']     = '1100'
os.environ['METHOD_TIMEOUT']   = '900'
os.environ['PYSR_PARALLELISM'] = 'multithreading'

# ONE_EQUATION must be UNSET for full runs
os.environ.pop('ONE_EQUATION', None)

# Feynman / Nguyen / noise — FULL counts
os.environ['N_FEYNMAN_TASKS']   = '30'
os.environ['N_NGUYEN_TASKS']    = '12'

# DeFi benchmark module-level constants
os.environ['_METHOD_TIMEOUT_SECS'] = '900'
os.environ['_PYSR_TIMEOUT_SECS']   = '1100'

# v7: exp1b env vars
os.environ['DEFI_V3C_NO_TIMEOUT_FLAGS'] = '1'
os.environ['DEFI_TASK_FILTER']          = 'portfolio'
os.environ['DEFI_SEEDS']               = '42,99,123,777,2024'

# v7: exp3/exp3b env var
os.environ['SKIP_PKG_CHECK'] = '1'

# v7: suppA / extrap env vars
os.environ['SKIP_PERF_ANALYSIS']    = '1'
os.environ['HYPATIAX_CORE_OPTIONAL'] = '1'

# v7: verify env vars (FIX-VERIFY)
import pathlib as _pl

_repro = _pl.Path(os.environ.get('REPRO_ROOT', '.'))
os.environ['PATCHED_DATA_DIR']   = str(_repro / 'hypatiax' / 'data' / 'results')
os.environ['VERIFY_RESULTS_DIR'] = str(_repro / 'hypatiax' / 'data' / 'results')

# v7: tables env vars (FIX-TABLES)
os.environ['TABLE_OUTDIR'] = str(_repro / 'hypatiax' / 'data' / 'results' / 'tables')

print('Config — FULL (paper-quality, FAST=0)')
print(f'  PySR  : {os.environ["PYSR_NITERATIONS"]} iters · {os.environ["PYSR_POPULATIONS"]} pops · {os.environ["PYSR_TIMEOUT"]}s/eq · {os.environ["METHOD_TIMEOUT"]}s/method')
print(f'  Tasks : DeFi={os.environ["N_TASKS_DEFI"]} · Feynman={os.environ["N_FEYNMAN_TASKS"]} · Nguyen={os.environ["N_NGUYEN_TASKS"]} · Instability={os.environ["N_TASKS_INSTABILITY"]}')
print(f'  NN    : {os.environ["NN_TIME_LIMIT"]}s limit · LLM retries={os.environ["LLM_RETRIES"]}')
print(f'  Model : {os.environ["LLM_MODEL"]}')
print('  Paper-quality mode — results WILL match paper targets.')

# Ensure output subdirectory tree exists
import pathlib

_results = pathlib.Path(os.environ.get('REPRO_ROOT', '.')) / 'hypatiax' / 'data' / 'results'
for _sub in [
    'comparison_results/extrapolation', 'comparison_results/feynman-tests/noise-sweep',
    'comparison_results/noise-noiseless/noiseless', 'comparison_results/noise-noiseless/15',
    'extrapolation', 'hybrid_llm_nn/all_domains', 'hybrid_llm_nn/defi',
    'hybrid_pysr/all_domains', 'hybrid_pysr/defi', 'llm_guided/all_domains',
    'llm_guided/defi', 'standalone_llm_nn', 'figures', 'tables',
]:
    (_results / _sub).mkdir(parents=True, exist_ok=True)
print('Output subdirs ready:', _results)


## Kaggle: Checkpoint / Resume Helper
> **Kaggle only** — tracks finished experiments so you can resume after the 12-h session restarts.
> Safe to skip on Colab Pro+.


In [ ]:
import datetime
import json
import os
import pathlib

CHECKPOINT_FILE = (
    pathlib.Path('/kaggle/working/hypatiax_checkpoint.json')
    if os.path.exists('/kaggle')
    else pathlib.Path('hypatiax_checkpoint.json')
)

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        return json.loads(CHECKPOINT_FILE.read_text())
    return {'completed': [], 'started': None}

def save_checkpoint(exp_name):
    cp = load_checkpoint()
    if exp_name not in cp['completed']:
        cp['completed'].append(exp_name)
    cp['last_updated'] = datetime.datetime.now().isoformat()
    CHECKPOINT_FILE.write_text(json.dumps(cp, indent=2))
    print(f'  Checkpoint saved: {exp_name}')

def is_done(exp_name):
    return exp_name in load_checkpoint()['completed']

cp = load_checkpoint()
print('Checkpoint file:', CHECKPOINT_FILE)
print('Completed:', cp.get('completed', []) or '(none yet)')
print()
print('Usage after each experiment: save_checkpoint("exp1")')
print('Skip if done: is_done("exp1")')


## 3 · Apply patches
Must run **before any benchmark**. Applies:
- **FIX-C1** — rename duplicate DeFi case names
- **FIX-C2** — replace any stray `hybrid_system_v40` references with `hybrid_system_v50_2`
- **FIX-T1/T2/XR3** — paper text corrections


In [ ]:
import os
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
for _script in [
    'scripts/patches/generate_patches.py',
    'scripts/patches/apply_patches.py',
]:
    r = subprocess.run([sys.executable, _script], cwd=_cwd)
    if r.returncode != 0:
        raise RuntimeError(f'{_script} failed')
# Verify patches applied correctly
r = subprocess.run([sys.executable, 'scripts/patches/apply_patches.py', '--verify'],
                   cwd=_cwd)
print('✓ Patches applied and verified' if r.returncode == 0 else '✗ Patch verify failed')


## 4 · Verify all imports resolve


In [ ]:
import importlib
import sys

importlib.invalidate_caches()

# Clear stale cached modules so fixes take effect
for m in [k for k in sys.modules if k.startswith(("hypatiax", "protocols", "core", "shared"))]:
    del sys.modules[m]

checks = {
    # Repro harness
    "protocols._base"                                  : "protocols._base",
    "protocols.universal_protocol"                     : "protocols.universal_protocol",
    "core.runners.common"                              : "core.runners.common",
    # Engine
    "hybrid_system_v50_2"                              : "hypatiax.tools.symbolic.hybrid_system_v50_2",
    "symbolic_engine"                                  : "hypatiax.tools.symbolic.symbolic_engine",
    "physics_aware_regressor"                          : "hypatiax.tools.symbolic.physics_aware_regressor",
    "smart_structure_detector"                         : "hypatiax.tools.symbolic.smart_structure_detector",
    # Validation
    "dimensional_validator"                            : "hypatiax.tools.validation.dimensional_validator",
    "domain_validator"                                 : "hypatiax.tools.validation.domain_validator",
    "ensemble_validator"                               : "hypatiax.tools.validation.ensemble_validator",
    "symbolic_validator"                               : "hypatiax.tools.validation.symbolic_validator",
    # Protocol scripts (authoritative runners)
    "experiment_protocol_ablation_exp1"                : "protocols.experiment_protocol_ablation_exp1",
    "experiment_protocol_defi_v3"                      : "protocols.experiment_protocol_defi_v3",
    "experiment_protocol_feynman_exp2"                 : "protocols.experiment_protocol_feynman_exp2",
    "experiment_protocol_nguyen12_exp3"                : "protocols.experiment_protocol_nguyen12_exp3",
    "experiment_protocol_noise_sweep"                  : "protocols.experiment_protocol_noise_sweep",
    "experiment_protocol_hybrid_routing"               : "protocols.experiment_protocol_hybrid_routing",
    "experiment_protocol_instability_rf02_04"          : "protocols.experiment_protocol_instability_rf02_04",
    "experiment_protocol_extrapolation_comparative"    : "protocols.experiment_protocol_extrapolation_comparative",
    "experiment_protocol_provenance_audit"             : "protocols.experiment_protocol_provenance_audit",
    # hypatiax/protocols/ — input-data/library modules (imported by root protocols/)
    "experiment_protocol_defi"           : "hypatiax.protocols.experiment_protocol_defi",
    "experiment_protocol_defi_20"        : "hypatiax.protocols.experiment_protocol_defi_20",
    "experiment_protocol_nguyen12"       : "hypatiax.protocols.experiment_protocol_nguyen12",
    "experiment_protocol_all_18_a"       : "hypatiax.protocols.experiment_protocol_all_18_a",
    "experiment_protocol_all_20"         : "hypatiax.protocols.experiment_protocol_all_20",
    "experiment_protocol_all_30"         : "hypatiax.protocols.experiment_protocol_all_30",
    "experiment_protocol_benchmark"      : "hypatiax.protocols.experiment_protocol_benchmark",
    "experiment_protocol_benchmark_v2"   : "hypatiax.protocols.experiment_protocol_benchmark_v2",
    "experiment_protocol_comparative"    : "hypatiax.protocols.experiment_protocol_comparative",
    # Support
    "extrapolation_test_protocol"                      : "hypatiax.experiments.tests.extrapolation_test_protocol",
    # Analysis & baselines
    "statistical_analysis"                     : "hypatiax.analysis.statistical_analysis",
    "baseline_neural_network"                          : "hypatiax.core.training.baseline_neural_network",
    "baseline_pure_llm"                                : "hypatiax.core.base_pure_llm.baseline_pure_llm",
}

all_ok = True
for label, dotted in checks.items():
    try:
        importlib.import_module(dotted)
        print(f"  ✓  {label}")
    except Exception as e:
        print(f"  ✗  {label:45s}  ← {e}")
        all_ok = False

print()
print("All imports OK ✓" if all_ok else "☢  Fix errors above before running experiments.")


## 5 · Exp 1 — DeFi 74-task benchmark (v3.0)
**Expected:** 89.2 % near-perfect (R²>0.99), 0 catastrophic failures\n**Wall time:** 2–4 h (K=1)


In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
# Clear stale locks
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_ablation_exp1.py'], cwd=_cwd)
if r.returncode != 0: raise RuntimeError('exp1 failed')
print('✓ Exp 1 done')


### §4 · Numerical issues check
> Run after any experiment. Scans all result JSON files for `inf`, `nan`, R²<−1, or negative RMSE.

In [ ]:
# §4 · Numerical issues check — scan all result JSON files for inf/nan/negative R²
import json
import math
import os
import pathlib

_results = pathlib.Path(os.environ.get('REPRO_ROOT', '.')) / 'hypatiax' / 'data' / 'results'
_issues  = []
_scanned = 0

def _bad(v):
    try:
        f = float(v)
        return math.isinf(f) or math.isnan(f)
    except Exception:
        return False

def _walk(obj, path=''):
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            kl = k.lower()
            if any(m in kl for m in ('rmse','r2','r_squared','mse','mae','score','loss')):
                if _bad(v):
                    _issues.append({'key': p, 'value': v})
                elif isinstance(v, (int,float)):
                    fv = float(v)
                    if ('r2' in kl or 'r_squared' in kl) and fv < -1.0:
                        _issues.append({'key': p, 'value': fv, 'note': 'R²<-1'})
                    if ('rmse' in kl or 'mse' in kl) and fv < 0:
                        _issues.append({'key': p, 'value': fv, 'note': 'negative'})
            _walk(v, p)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            _walk(item, f'{path}[{i}]')

for jf in _results.rglob('*.json'):
    try:
        _walk(json.loads(jf.read_text(errors='replace')))
        _scanned += 1
    except Exception:
        pass

print(f'Scanned {_scanned} JSON result files')
if _issues:
    print(f'⚠  {len(_issues)} numerical issue(s):')
    for iss in _issues[:20]:
        print(f'   {iss["key"]} = {iss["value"]} {iss.get("note","")}')
else:
    print('✓ No RMSE=inf / R²=inf / R²<-1 / negative RMSE found')


## 6 · Exp 1b — Portfolio Variance seed sweep  *(§10.5)*
**Expected:** P(HypatiaX > PureLLM) ≈ 0.76 across seeds [42, 99, 123, 777, 2024]  
**Wall time:** ~20–40 min

In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_defi_v3.py',
    '--task', 'portfolio_variance', '--seeds', '42', '99', '123', '777', '2024'],
    cwd=_cwd)
if r.returncode != 0: raise RuntimeError('exp1b failed')
print('✓ Exp 1b done')


## 7 · Exp 2 — Feynman 30-equation extrapolation  *(§10.7)*
**Expected:** 9/30 (30 %) · 40/60 PCA split · 3 hardware runs (RF-09); Kaggle 4-vCPU is primary  
**Wall time:** 4–8 h  *(skip with `--skip-slow` in `run_all.sh`)*

In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_feynman_exp2.py'], cwd=_cwd)
if r.returncode != 0: raise RuntimeError('exp2 failed')
print('✓ Exp 2 done')


## 8 · Exp 3 — Nguyen-12 standard SR suite  *(§10.8 — primary run)*
**Expected:** 11/12 H (91.7 %) · 10/12 P (83.3 %) · 0/12 NN · MW P>NN U=113, p=0.0097  
**Wall time:** 30–90 min  ·  **SEED=42** (fixed for reproducibility)

In [ ]:
# ── Exp 3 · Nguyen-12 SR suite (§10.8 primary)  SEED=42 ─────────────────
# FIX: replaced Jupyter !python magic with subprocess (safe for .py execution)
import os
import pathlib
import subprocess
import sys

# Clear stale protocol cache lock
_results_dir = pathlib.Path(os.environ.get("REPRO_ROOT", ".")) / "hypatiax" / "data" / "results"
_locks = list(_results_dir.glob(".lock_*")) if _results_dir.exists() else []
if _locks:
    for _l in _locks: _l.unlink()
    print(f"  Cleared {len(_locks)} stale lock(s) — experiment will run fresh")
else:
    print("  No stale locks found")

# Run via subprocess so it works both in Jupyter and as a plain .py script
_result = subprocess.run(
    [sys.executable, "protocols/experiment_protocol_nguyen12_exp3.py"],
    cwd=os.environ.get("REPRO_ROOT", "."),
)
if _result.returncode != 0:
    raise RuntimeError(f"Exp 3 failed with returncode {_result.returncode}")
print("  ✓ Exp 3 done")


### 8b · Exp 3b — Nguyen-12 stability check  *(§10.8 — seed 123)*
**Expected:** result consistent with SEED=42 run  
**Wall time:** 30–90 min

In [ ]:
# ── Exp 3b · Nguyen-12 stability check (§10.8)  SEED=123 ────────────────
# FIX: replaced Jupyter !python magic with subprocess
import os
import pathlib
import subprocess
import sys

# Clear stale lock for exp3b
_results_dir = pathlib.Path(os.environ.get("REPRO_ROOT", ".")) / "hypatiax" / "data" / "results"
for _l in list(_results_dir.glob(".lock_*")) if _results_dir.exists() else []:
    _l.unlink()

_result = subprocess.run(
    [sys.executable, "protocols/experiment_protocol_nguyen12_exp3.py", "--seed", "123"],
    cwd=os.environ.get("REPRO_ROOT", "."),
)
if _result.returncode != 0:
    raise RuntimeError(f"Exp 3b failed with returncode {_result.returncode}")
print("  ✓ Exp 3b done")


## 9 · Supp B — Noise & sample-complexity sweep
**Expected:** EHD 100 % at all σ levels; plateau ~N=500\n**Wall time:** 6–12 h


In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_noise_sweep.py'], cwd=_cwd)
if r.returncode != 0: raise RuntimeError('suppB failed')
print('✓ Supp B done')


## 10 · Supp A — Hybrid routing improvements  *(Supp A §1–8)*
**Expected:** +6 pp Fix1 (extrapolation probe) · +5 pp Fix2 (formula complexity) · +1 pp Fix3 (LLM feature aug)  
Override priority: Fix2 > Fix1 > Fix3 > Fix4  
**Wall time:** ~30–60 min

In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_hybrid_routing.py'], cwd=_cwd)
if r.returncode != 0: raise RuntimeError('suppA failed')
print('✓ Supp A done')


## 11 · §10.9 — Instability analysis (RF02/04)
**Expected:** Spearman ρ=−0.70 (p<0.001) · 70 tasks × K=30 runs · C-Collapse Portfolio ES anomaly (RF-06)  
**Wall time:** 3–6 h  (K=30 full sweep)  

> ⚠️ **The cell below sets `LLM_K_RUNS=30` for the full sweep.**  
> Set `LLM_K_RUNS=1` in cell 2 (Runtime config) for a smoke-test only.

In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
# Use K_RUNS=30 for full instability sweep; FAST mode already sets N_TASKS low
env = {**os.environ, 'LLM_K_RUNS': '30'}
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_instability_rf02_04.py'],
    cwd=_cwd, env=env)
if r.returncode != 0: raise RuntimeError('instability failed')
os.environ['LLM_K_RUNS'] = '1'  # restore
print('✓ Instability done')


## 12 · §10.8 — Extrapolation comparative
**Expected:** cross-method R² comparison across near/medium/far OOD regimes  
**Wall time:** ~20–40 min

In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_rd = pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results'
for _l in list(_rd.glob('.lock_*')) if _rd.exists() else []: _l.unlink()
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_extrapolation_comparative.py'], cwd=_cwd)
if r.returncode != 0: raise RuntimeError('extrap failed')
print('✓ Extrap done')


## 13 · §11 — Provenance audit
**Expected:** Run after all experiments complete


In [ ]:
import os
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
r = subprocess.run([sys.executable,
    'protocols/experiment_protocol_provenance_audit.py'], cwd=_cwd)
# Non-blocking: provenance_audit.py may be absent in the public repo.
# universal_protocol.py now returns a failure dict instead of sys.exit(1),
# so a non-zero returncode here is a warning, not a pipeline killer.
if r.returncode != 0:
    print(f'⚠  Provenance audit exited {r.returncode} — check logs/provenance_audit/.')
    print('   This is non-blocking. All experiment results are still valid.')
else:
    print('✓ Provenance done')


In [ ]:
# §11b — Link every result file to its source family and paper section
# Reads provenance_map.json (optional); outputs logs/provenance_audit/
import os
import pathlib
import subprocess

REPO_BASE = pathlib.Path(os.environ["REPRO_ROOT"])
pmap      = REPO_BASE / "provenance_map.json"

if not pmap.exists():
    print(f"⚠  provenance_map.json not found at {pmap} — skipping §11b.")
    print("   To generate it, run:  python discover_provenance.py --generate-map")
else:
    out_dir = REPO_BASE / "logs" / "provenance_audit"
    out_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["python3", str(REPO_BASE / "discover_provenance.py"),
         "--root", str(REPO_BASE),
         "--map",  str(pmap),
         "--out",  str(out_dir)],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:])  # tail — full output in logs/provenance_audit/
    if result.returncode != 0:
        print(result.stderr)


In [ ]:
# §11c — Build internal import DAG for reproducibility verification
# Outputs logs/repro_output/import_graph.{dot,png,pdf}
import os
import pathlib
import subprocess

REPO_BASE = pathlib.Path(os.environ["REPRO_ROOT"])
out_dir   = REPO_BASE / "logs" / "repro_output"
out_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    ["python3", str(REPO_BASE / "scan_internal_imports.py"),
     "--root", str(REPO_BASE),
     "--out",  str(out_dir)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
# Import graph rendered to logs/repro_output/import_graph.{dot,png,pdf}


## 13b · Pipeline smoke-test (post-§11 summary)
> Prints a single PASS / FAIL banner after all §11 provenance and import-graph
> cells have run. A PASS here means: all 5 patches applied, 0 import issues,
> provenance map present (or gracefully skipped), and the import DAG written.


In [ ]:
# ── Pipeline smoke-test ───────────────────────────────────────────────────
import os
import pathlib
import subprocess
import sys

REPO_BASE = pathlib.Path(os.environ.get("REPRO_ROOT", "."))
checks = {}

# 1. Patches fully applied
r = subprocess.run(
    [sys.executable, "scripts/patches/apply_patches.py", "--verify"],
    capture_output=True, text=True, cwd=REPO_BASE
)
checks["patches"] = (r.returncode == 0 and "All 5 patch(es) applied" in r.stdout)

# 2. Import scan clean
import_report = REPO_BASE / "logs" / "repro_output" / "import_report.txt"
if import_report.exists():
    txt = import_report.read_text()
    checks["imports"] = all(
        f"{lbl}: 0" in txt
        for lbl in ["[A] Stale engine imports", "[B] Ghost imports",
                     "[C] Protocol layer leaks", "[D] Import cycles"]
    )
else:
    checks["imports"] = False

# 3. Import DAG written
checks["import_dag"] = (REPO_BASE / "logs" / "repro_output" / "import_graph.dot").exists()

# 4. Provenance map present or gracefully absent (warn only)
pmap = REPO_BASE / "provenance_map.json"
checks["provenance_map"] = pmap.exists()  # warning, not blocking

# ── Print results ─────────────────────────────────────────────────────────
print("\n── Smoke-test results ──────────────────────────────────")
blocking_pass = True
for name, ok in checks.items():
    warn_only = name == "provenance_map"
    symbol = "✓" if ok else ("⚠" if warn_only else "✗")
    print(f"  {symbol}  {name}")
    if not ok and not warn_only:
        blocking_pass = False

print()
if blocking_pass:
    print("✅  PASS — pipeline is clean. Proceed to Step 14 (validate metrics).")
else:
    print("❌  FAIL — fix issues above before validating metrics.")
    raise SystemExit(1)


### §7 · Reproducibility gate
> Calls `provenance_audit.py` for the full Yes/No verdict with blocker list.

In [ ]:
# §7 · Reproducibility gate — provenance_audit.py
# Run AFTER all experiments and §11 cells.
import json as _json
import os
import pathlib
import subprocess
import sys

_cwd  = os.environ.get('REPRO_ROOT', '.')
_seed = int(os.environ.get('NN_SEED', '42'))
_fast = int(os.environ.get('FAST', '0'))

_audit = pathlib.Path(_cwd) / 'provenance_audit.py'
if not _audit.exists():
    print(f'⚠  provenance_audit.py not found at {_audit}')
    print('   Drop it at repo root next to run_all.py and re-run this cell.')
else:
    _out = pathlib.Path(_cwd) / 'logs' / 'provenance_audit_report.json'
    r = subprocess.run(
        [sys.executable, str(_audit),
         '--root', _cwd,
         '--seed', str(_seed),
         '--json', '--out', str(_out)],
        cwd=_cwd
    )
    if _out.exists():
        report = _json.loads(_out.read_text())
        repro  = report.get('reproducible', False)
        blocks = report.get('blocker_count', 0)
        print()
        print('══════════════════════════════════════════')
        if repro:
            print('  REPRODUCIBLE: Yes ✓')
        else:
            print(f'  REPRODUCIBLE: No ✗   ({blocks} blocker(s))')
            for f in report.get('findings', []):
                if f['level'] == 'BLOCK':
                    print(f'    [{f["section"]}] {f["message"]}')
        print('══════════════════════════════════════════')
        if not repro and not _fast:
            raise SystemExit('Reproducibility gate failed — fix blockers before submitting.')
        elif not repro and _fast:
            print('  (FAST mode — gate is advisory, not blocking)')
    else:
        print('⚠  JSON report not written — check provenance_audit.py output above')


## 14 · Validate metrics against paper

| Metric | Paper target | Section |
|---|---|---|
| R²>0.99 rate (DeFi 74-task)          | 89.2 %       | §10.2 |
| Catastrophic failures                | 0            | §10.2 |
| Speedup (LLM-routed cases)           | 1.73×        | §10.4 |
| Portfolio Variance P(H>P)            | ≈ 0.76       | §10.5 |
| Ablation MW U (Run A)                | 126, p=0.295 | §10.6 |
| Feynman success rate                 | 9/30 (30 %)  | §10.7 |
| Nguyen-12 HypatiaX rate              | 11/12 (91.7%)| §10.8 |
| Instability Spearman ρ               | −0.70, p<0.001| §10.9 |

In [ ]:
import os
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
r = subprocess.run([sys.executable,
    'scripts/patches/verify_results.py', '--report', '--json'], cwd=_cwd)
r2 = subprocess.run([sys.executable,
    'reproducibility/hash_lock.py', '--check'], cwd=_cwd)
if r.returncode != 0 or r2.returncode != 0:
    raise RuntimeError('Verification failed')
print('✓ Verification done')


## 15 · Generate figures and tables


In [ ]:
import os
import pathlib
import subprocess
import sys

_cwd = os.environ.get('REPRO_ROOT', '.')
_results = str(pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results')
_tbl_dir = str(pathlib.Path(_results) / 'tables')

# Figures → hypatiax/data/results/figures/
r = subprocess.run([sys.executable, 'figures/generate_figures.py',
                    '--outdir', f'{_results}/figures'], cwd=_cwd)
if r.returncode != 0:
    print('  ⚠ generate_figures.py failed (non-blocking)')

# Tables → hypatiax/data/results/tables/
# v7: FIX-TABLES — pass TABLE_OUTDIR + VERIFY_RESULTS_DIR so generate_tables.py
# writes to the correct canonical location (not paper/tables/).
_env_t = {**os.environ,
           'TABLE_OUTDIR': _tbl_dir,
           'VERIFY_RESULTS_DIR': _results}
r2 = subprocess.run([sys.executable, 'scripts/patches/generate_tables.py',
                     '--outdir', _tbl_dir], cwd=_cwd, env=_env_t)
if r2.returncode != 0:
    print('  ⚠ generate_tables.py failed (non-blocking)')

# Report counts
fig_dir = pathlib.Path(_results) / 'figures'
tbl_dir = pathlib.Path(_results) / 'tables'
# v7: FIX-INVENTORY — also check paper/tables/ as fallback
if not tbl_dir.exists() or not any(tbl_dir.glob('*.tex')):
    tbl_dir = pathlib.Path(_cwd) / 'paper' / 'tables'
n_pdf = len(list(fig_dir.glob('*.pdf'))) if fig_dir.exists() else 0
n_tex = len(list(tbl_dir.glob('*.tex'))) if tbl_dir.exists() else 0
print(f'✓ Figures: {n_pdf} PDF(s) in {fig_dir}')
print(f'✓ Tables : {n_tex} TeX(s) in {tbl_dir}')


## 16 · Compile the paper (optional)
Three `pdflatex` passes required for cross-references and bibliography.


In [ ]:
import os
import pathlib
import subprocess

_cwd = os.path.join(os.environ.get('REPRO_ROOT', '.'), 'paper')
if not pathlib.Path(_cwd).exists():
    print('  ⚠ paper/ directory not found — skipping LaTeX build')
else:
    for _tex in ['jmlr-hypatiax-full-paper.tex',
                 'supp_routing_improvements.tex',
                 'supp_benchmark_report.tex']:
        if not pathlib.Path(_cwd, _tex).exists():
            print(f'  ⚠ {_tex} not found — skipping')
            continue
        stem = _tex.replace('.tex', '')
        for _cmd in [
            ['pdflatex', _tex],
            ['bibtex',   stem],
            ['pdflatex', _tex],
            ['pdflatex', _tex],
        ]:
            subprocess.run(_cmd, cwd=_cwd)
    print('✓ LaTeX build done')


---
## §12 · Paper audit notebooks (NB-01 to NB-06)

These six notebooks scan `jmlr-hypatiax-paper-final.tex` (NB-01–05) and the
benchmark `.py` files (NB-06). They are **diagnostic tools**: they exit 0
and surface issues in their output cells — they do not auto-fix.

| Notebook | Scans | Flags | Fix IDs |
|---|---|---|---|
| NB-01 | `.tex` bibliography | Missing/duplicate bibitems | FIX-B1, B2, B3 |
| NB-02 | `.tex` labels/refs | Undefined refs, dup labels, Supp A cross-refs | FIX-XR1–XR4 |
| NB-03 | `.tex` structure | Section tree, equation count | (diagnostic) |
| NB-04 | `.tex` numbers | 70 vs 71, five-stage vs five-layer | FIX-N1, N2, N3 |
| NB-05 | `.tex` + `figures/` | Missing image files, fbox placeholders | FIX-F1–F4 |
| NB-06 | `.py` source files | Dup case names, stale v40, split mismatch, exposed keys | FIX-C1, C3 |

> **NB-06** is also run as a **preflight check** at pipeline start (before Phase 1 experiments).
> **NB-01–05** are run in **Phase 4-B** (after figures/tables, before pdflatex).
> All notebooks must be in `notebooks/` and the `.tex` file must be in `paper/`.

In [ ]:
# §12 — Execute all 6 audit notebooks via subprocess
# NB-01–05 need the main .tex file copied beside them in notebooks/
import os
import pathlib
import shutil
import subprocess

REPO_BASE = pathlib.Path(os.environ["REPRO_ROOT"])
NOTEBOOKS = REPO_BASE / "notebooks"

# Copy the .tex file if paper/ dir exists (may be absent in the public repo)
paper_dir = REPO_BASE / "paper"
if paper_dir.exists():
    TEX_CANDIDATES = list(paper_dir.glob("jmlr-hypatiax*.tex"))
    if TEX_CANDIDATES:
        shutil.copy(TEX_CANDIDATES[0], NOTEBOOKS / TEX_CANDIDATES[0].name)
        print(f"Copied: {TEX_CANDIDATES[0].name} → notebooks/")
    else:
        print("WARNING: no jmlr-hypatiax*.tex found in paper/ — NB-01–05 will warn at Step 1")
else:
    print("INFO: paper/ directory not present in this repo layout.")
    print("      NB-01–05 scan .tex files — they will skip gracefully if none is found.")

AUDIT_NBS = [
    "NB-06_Code_Quality_Pipeline_Integrity.ipynb",  # code-quality preflight first
    "NB-01_Citation_Bibliography_Audit.ipynb",
    "NB-02_CrossReference_Label_Audit.ipynb",
    "NB-03_Section_Structure_Numbering.ipynb",
    "NB-04_Numerical_Consistency_Checker.ipynb",
    "NB-05_Figure_Image_Dependency_Checker.ipynb",
]

for nb_name in AUDIT_NBS:
    nb_path = NOTEBOOKS / nb_name
    if not nb_path.exists():
        print(f"  SKIP (not found): {nb_name}")
        continue
    r = subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook", "--execute",
         "--inplace", "--ExecutePreprocessor.timeout=300", str(nb_path)],
        capture_output=True, text=True
    )
    status = "✓" if r.returncode == 0 else "✗"
    print(f"  {status}  {nb_name}")
    if r.returncode != 0:
        print(f"      {r.stderr[-300:]}")

print("\nAudit complete. Open each notebooks/NB-0*.ipynb to review findings.")
